# Create ITSM Ticket Survival Dataset

This notebook loads the cleaned ITSM dataset and metadata table using `dd_cleaner.notebook_utils`, reads package discovery APIs from `featurization` and `kmds_modeling`, and constructs a ticket survival dataset for closure event analysis.

In [ ]:
import pandas as pd
from pathlib import Path

from dd_cleaner.notebook_utils import init_notebook_session, get_cleaned_data, get_metadata_table
from featurization import get_package_info
from kmds_modeling import get_spec_questions

# Package discovery and modeling spec helper examples
package_info = get_package_info()
requirements = {
    "project": {
        "task_type": "KAPLAN_MEIER"
    }
}
spec_questions = get_spec_questions(requirements)

package_info, spec_questions


({'package_name': 'kmds-featurization',
  'version': '0.3.2',
  'entry_point': 'featurization-cli',
  'cli_commands': ['init', 'bootstrap', 'run', 'advise', 'check', 'add-stage'],
  'documentation_note': 'This package no longer ships internal docs in the installed package. Use the repository top-level documents/ folder for onboarding and implementation guidance.',
  'usage': {'overview': 'Use this package by configuring a workspace in featurizer_config.yaml, then running the CLI from that workspace. For cross-sectional workflows, follow the cross-sectional guide; for event-log survival workflows, follow the survival guide.',
   'cross_sectional_guide': 'documents/user_guide_cs_featurization.md',
   'survival_guide': 'documents/survival_featurization_pipeline.md',
   'guidance': 'Feature selection is recommended for wide and short datasets. This package does not provide a dedicated featurization pipeline for wide and short datasets.'}},
 [{'field': 'project.strategy',
   'question': 'Ch

In [16]:
workspace_dir = Path.cwd()
project_root = workspace_dir if (workspace_dir / 'config.yaml').exists() else workspace_dir.parent
coord, artifacts_df = init_notebook_session(str(project_root))
metadata_df = get_metadata_table(coord)
cleaned_df = get_cleaned_data(coord)

artifacts_df, metadata_df.head(), cleaned_df.head()


✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/itsm_analysis

Available Artifacts:

Artifact Name  \
0                         Raw Data   
1                     Cleaned Data   
2                User Cleaned Data   
3             Tagged Entities (DD)   
4  Cleaning Recommendations Report   
5                 Profiling Report   
6                   Handshake File   
7                  Quarantine File   
8               Metadata Authority   

                                         File Name  \
0                          itsm_group_analysis.csv   
1                    itsm_group_analysis_clean.csv   
2             itsm_group_analysis_user_cleaned.csv   
3         itsm_group_analysis_analysis_results.csv   
4                      cleaning_recommendations.md   
5          itsm_group_analysis_profiling_report.md   
6  itsm_group_analysis_parser_cleaner_handshake.md   
7               itsm_group_analysis_quarantine.csv   
8           itsm_group_analysis_metadata_table.csv   

                                            Location  Exists  
0                       data/itsm_group_analysis.csv    True  
1      data/dd_cleaner/itsm_group_analysis_clean.csv    True  
2  data/dd_cleaner/itsm_group_analysis_user_clean...   False  
3  documents/dd_analysis_results/itsm_group_analy...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/itsm_group_analysis_profi...    True  
6  documents/dd_cleaner/itsm_group_analysis_parse...    True  
7  data/quarantine/itsm_group_analysis_quarantine...   False  
8  data/dd_cleaner/itsm_group_analysis_metadata_t...   False

(                     Artifact Name  \
 0                         Raw Data   
 1                     Cleaned Data   
 2                User Cleaned Data   
 3             Tagged Entities (DD)   
 4  Cleaning Recommendations Report   
 5                 Profiling Report   
 6                   Handshake File   
 7                  Quarantine File   
 8               Metadata Authority   
 
                                          File Name  \
 0                          itsm_group_analysis.csv   
 1                    itsm_group_analysis_clean.csv   
 2             itsm_group_analysis_user_cleaned.csv   
 3         itsm_group_analysis_analysis_results.csv   
 4                      cleaning_recommendations.md   
 5          itsm_group_analysis_profiling_report.md   
 6  itsm_group_analysis_parser_cleaner_handshake.md   
 7               itsm_group_analysis_quarantine.csv   
 8           itsm_group_analysis_metadata_table.csv   
 
                                             Location  E

## Build the survival dataset

The event of interest is ticket closure as indicated by `incident_state == 'closed'`. The observation window ends at the last closed ticket and begins one year prior. Closed tickets receive a duration computed from `opened_at` to `closed_at`.

In [17]:
df = cleaned_df.copy()

for col in ['opened_at', 'closed_at']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'incident_state' not in df.columns or 'number' not in df.columns:
    raise KeyError('Expected cleaned dataset to contain at least `incident_state` and `number` columns.')

df['incident_state'] = df['incident_state'].astype(str).str.lower()
closed_states = {'closed'}
df['survival_event'] = df['incident_state'].isin(closed_states).astype(int)

df['duration_days'] = (df['closed_at'] - df['opened_at']).dt.total_seconds() / 86400.0

observation_end = df.loc[df['incident_state'] == 'closed', 'closed_at'].max()
if pd.isna(observation_end):
    observation_end = df['opened_at'].max()
observation_start = observation_end - pd.Timedelta(days=365)

survival_df = df[df['opened_at'] >= observation_start].copy()

# Use the last observed record for each ticket number within the observation window
survival_df = survival_df.sort_values(['number', 'closed_at'], ascending=[True, True]).groupby('number', sort=False).tail(1).copy()
survival_df['subject_id'] = survival_df['number']

# Remove records with unknown assignment group before filtering by closure count
survival_df = survival_df[survival_df['assignment_group'] != '?'].copy()

MIN_CLOSURE_THRESHOLD = 20
if 'assignment_group' not in survival_df.columns:
    raise KeyError('Expected cleaned dataset to contain `assignment_group` for support group filtering.')

closure_counts = (
    survival_df[survival_df['survival_event'] == 1]
    .groupby('assignment_group')
    .size()
)
keep_groups = closure_counts[closure_counts > MIN_CLOSURE_THRESHOLD].index.tolist()
survival_df = survival_df[survival_df['assignment_group'].isin(keep_groups)].copy()

save_path = Path(coord.working_dir) / 'data' / 'dd_cleaner' / 'itsm_ticket_survival_dataset.csv'
save_path.parent.mkdir(parents=True, exist_ok=True)
survival_df.to_csv(save_path, index=False)

save_path, len(survival_df), survival_df[['subject_id', 'assignment_group', 'incident_state', 'survival_event', 'duration_days']].head()


(PosixPath('/home/rajiv/programming/kmds_migration/itsm_analysis/data/dd_cleaner/itsm_ticket_survival_dataset.csv'),
 22652,
     subject_id assignment_group incident_state  survival_event  duration_days
 3   INC0000045         Group 56         closed               1       5.447222
 12  INC0000047         Group 24         closed               1       6.222222
 19  INC0000057         Group 70         closed               1       5.868056
 23  INC0000060         Group 25         closed               1       7.265278
 31  INC0000062         Group 23         closed               1       5.376389)

In [18]:
# Show incident states that are not closed
non_closed_df = cleaned_df[cleaned_df['incident_state'].astype(str).str.lower() != 'closed']
non_closed_counts = non_closed_df['incident_state'].astype(str).value_counts(dropna=False)
non_closed_counts


incident_state
Active                38716
New                   36407
Resolved              25751
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

In [19]:
# Show count of closed tickets
closed_count = (cleaned_df['incident_state'].astype(str).str.lower() == 'closed').sum()
closed_count


np.int64(24985)